# 2회차 실습: Norm(노름)·Dot product(내적)·Cosine similarity (MNIST)

> Part 1: 2회차 (Norm·Dot product·Cauchy-Schwarz·Cosine similarity)
> 사전 reading: Strang §1.2 / MML §3.1, §3.2 (참조) / 3Blue1Brown EoLA Ch.9

이 노트북은 강의교안 2회차의 흐름(B 섹션 Norm → C 섹션 Dot product·각도·Cauchy-Schwarz → D 섹션 Cosine similarity·MNIST)을 그대로 따라갑니다. 매 절은 **Definition → (Theorem) → Application** 순서로 정리합니다.

## 학습 목표

이번 실습이 끝나면 다음을 NumPy 코드로 직접 보일 수 있습니다.

1. $\ell_1, \ell_2, \ell_\infty$ Norm을 **직접 구현**하고 `np.linalg.norm`과 결과를 비교합니다.
2. Norm의 세 가지 성질 **(N1) 양정성(positive definiteness)·(N2) 동차성(absolute homogeneity)·(N3) 삼각부등식(triangle inequality)**을 수치로 검증합니다.
3. **Dot product**와 **Cosine similarity**를 직접 구현하고 스케일 불변성을 확인합니다.
4. **Cauchy-Schwarz 부등식** $|\mathbf{u}\cdot\mathbf{v}| \le \|\mathbf{u}\|\,\|\mathbf{v}\|$를 수치로 검증합니다.
5. MNIST 이미지(8×8 mini-MNIST)를 $\mathbb{R}^{64}$ Vector로 보고, 두 이미지 cosine similarity·평균 비교·Top-5 nearest neighbor를 계산합니다.

### 정의·정리될 객체 목록

| 번호 | 객체 |
|---|---|
| 정의 2.1 | Euclidean Norm ($\ell_2$) + 일반화 $\ell_p$ |
| 정의 2.2 | Dot product (Inner product) |
| 정리 2.1 | Cauchy-Schwarz 부등식 |
| 정의 2.3 | 각도 $\theta$ |
| 정의 2.4 | Orthogonal(직교) |
| 정의 2.5 | Cosine similarity |

### 사용 라이브러리

- NumPy, Matplotlib (필수)
- scikit-learn (MNIST 일부 로드용)

In [ ]:
# Colab 한글 폰트 설정 (matplotlib 깨짐 방지)
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # 한글 폰트 자동 등록 (NanumGothic)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
print('NumPy version:', np.__version__)

# 1회차에서 본 R^4의 두 Vector (오늘 Norm·Dot product에서 다시 사용)
a = np.array([3, 1, -2, 4], dtype=float)
b = np.array([-1, 2, 5, 0], dtype=float)
print('a =', a)
print('b =', b)

## 1. Definition: Norm (정의 2.1)

### 정의 2.1 (Euclidean Norm, $\ell_2$ Norm)
$\mathbf{x} = (x_1, \ldots, x_n)^\top \in \mathbb{R}^n$의 **Euclidean Norm**:
$$\|\mathbf{x}\|_2 := \sqrt{x_1^2 + x_2^2 + \cdots + x_n^2} = \sqrt{\sum_{i=1}^n x_i^2}$$

### 일반화: $\ell_p$ Norm ($p \ge 1$)
$$\|\mathbf{x}\|_p := \left(\sum_i |x_i|^p\right)^{1/p}, \qquad \|\mathbf{x}\|_\infty := \max_i |x_i|$$

**`np.linalg.norm`을 호출하지 않고** 직접 구현합니다.

In [ ]:
def my_norm(v, p=2):
    """p-노름 직접 구현. p=1, 2, np.inf 지원.
    NumPy 기본 연산만 사용 (np.linalg.norm 금지).
    """
    v = np.asarray(v, dtype=float)
    if p == 1:
        return float(np.abs(v).sum())
    if p == 2:
        return float(np.sqrt((v ** 2).sum()))
    if p == np.inf:
        return float(np.abs(v).max())
    # 일반 p
    return float((np.abs(v) ** p).sum() ** (1.0 / p))


# 검증: 직접 구현 vs 라이브러리
print('a =', a)
for p in [1, 2, np.inf]:
    mine = my_norm(a, p)
    lib = np.linalg.norm(a, p)
    print(f'  p={p}:  mine={mine:.6f}, lib={lib:.6f}, match={np.isclose(mine, lib)}')

# 강의교안 B-4의 예제 재현: 같은 a에서 세 잣대
print(f'\n같은 Vector a, 세 가지 크기:')
print(f'  ||a||_1   = {my_norm(a, 1):.4f}   (재료 총 개수)')
print(f'  ||a||_2   = {my_norm(a, 2):.4f}   (피타고라스 종합 크기)')
print(f'  ||a||_inf = {my_norm(a, np.inf):.4f}   (가장 큰 한 성분)')

## 2. Theorem: Norm의 세 가지 성질 (N1·N2·N3)

강의교안 B-3에서 본 사실: Euclidean Norm은 다음 세 가지를 만족합니다.

| 성질 | 식 | 의미 |
|---|---|---|
| (N1) **양정성** (positive definiteness) | $\|\mathbf{x}\| \ge 0$, $\|\mathbf{x}\| = 0 \iff \mathbf{x} = \mathbf{0}$ | 거리는 음수 X |
| (N2) **동차성** (absolute homogeneity) | $\|\alpha\mathbf{x}\| = \lvert\alpha\rvert\,\|\mathbf{x}\|$ | $\alpha$배 늘리면 길이도 $\lvert\alpha\rvert$배 |
| (N3) **삼각부등식** (triangle inequality) | $\|\mathbf{x} + \mathbf{y}\| \le \|\mathbf{x}\| + \|\mathbf{y}\|$ | 직진 ≤ 우회 |

(N1)·(N2)는 정의에서 직접 따라옵니다. (N3) 삼각부등식(triangle inequality)은 §4 Cauchy-Schwarz의 결과로 따라옵니다 (강의교안 C-5).

여기서는 세 성질을 임의 Vector로 **수치 검증**합니다.

In [ ]:
rng = np.random.default_rng(2026)
n_trials = 1000
n = 5

viol_N1 = viol_N2 = viol_N3 = 0
for _ in range(n_trials):
    x = rng.normal(size=n)
    y = rng.normal(size=n)
    alpha = rng.normal()
    
    # (N1) 양정성(positive definiteness)
    if my_norm(x, 2) < 0 or (np.allclose(x, 0) and my_norm(x, 2) != 0):
        viol_N1 += 1
    
    # (N2) 동차성(absolute homogeneity)
    if not np.isclose(my_norm(alpha * x, 2), abs(alpha) * my_norm(x, 2)):
        viol_N2 += 1
    
    # (N3) 삼각부등식(triangle inequality)
    if my_norm(x + y, 2) > my_norm(x, 2) + my_norm(y, 2) + 1e-10:
        viol_N3 += 1

print(f'시행 {n_trials}회, R^{n} 임의 Vector')
print(f'  (N1) 양정성(positive definiteness)   위반: {viol_N1}회')
print(f'  (N2) 동차성(absolute homogeneity)   위반: {viol_N2}회')
print(f'  (N3) 삼각부등식(triangle inequality) 위반: {viol_N3}회')
print('✓ 세 성질 모두 임의 Vector에서 성립 (수치 검증)')

## 3. 시각화: 단위구 $\{\mathbf{x} : \|\mathbf{x}\| = 1\}$

강의교안 B-5의 핵심 그림: $\mathbb{R}^2$의 단위구 모양이 Norm마다 다릅니다.

| Norm | 모양 |
|---|---|
| $\ell_1$ | 마름모 (꼭짓점이 좌표축 위) |
| $\ell_2$ | 원 |
| $\ell_\infty$ | 정사각형 |

같은 "크기 1"이지만 모양이 다릅니다: 어떤 Norm을 쓰느냐가 "어디까지가 가까운가"의 범위를 다르게 정의합니다.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
circle = np.stack([np.cos(theta), np.sin(theta)])  # 2 × 400, 단위 원

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, p, title in zip(axes, [1, 2, np.inf], ['L1 (마름모)', 'L2 (원)', 'L_inf (정사각형)']):
    # 단위 원의 각 점을 그 점의 p-노름으로 정규화 → 해당 p-노름 단위구의 점
    norms = np.array([my_norm(circle[:, i], p) for i in range(circle.shape[1])])
    pts = circle / norms
    ax.plot(pts[0], pts[1])
    ax.set_aspect('equal')
    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.grid(True, alpha=0.3)
    ax.set_title(title)
    ax.axhline(0, color='gray', linewidth=0.5)
    ax.axvline(0, color='gray', linewidth=0.5)
plt.suptitle('p-Norm의 단위구 ($\\mathbb{R}^2$)')
plt.tight_layout()
plt.show()

print('관찰: L1의 꼭짓점이 좌표축 위 → sparse 해 유도. L2는 모든 방향 균등.')

## 4. Definition: Dot product·각도 (정의 2.2·2.3)

### 정의 2.2 (Dot product)
$\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$의 **Dot product**(내적, inner product):
$$\mathbf{u} \cdot \mathbf{v} := u_1 v_1 + u_2 v_2 + \cdots + u_n v_n = \sum_{i=1}^n u_i v_i \;\in\; \mathbb{R}$$
다른 표기: $\mathbf{u}^\top \mathbf{v}$, $\langle \mathbf{u}, \mathbf{v} \rangle$.

### Norm과의 다리
$$\mathbf{u} \cdot \mathbf{u} = \sum u_i^2 = \|\mathbf{u}\|_2^2$$
→ **$\ell_2$ Norm은 Dot product에서 자동으로 유도**됩니다.

### 정의 2.3 (각도)
$\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ ($\ne \mathbf{0}$)에 대해 각 $\theta \in [0, \pi]$를
$$\cos\theta := \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|_2\,\|\mathbf{v}\|_2}$$
으로 **정의**합니다.

이 정의가 정당하려면 $\cos\theta \in [-1, 1]$이 보장되어야 합니다 → §5 Cauchy-Schwarz.

In [ ]:
def my_dot(u, v):
    """내적 직접 구현 (for문 없는 numpy 연산)"""
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float((u * v).sum())


# a, b의 Dot product
print(f'a · b           = {my_dot(a, b):.4f}')
print(f'np.dot(a, b)    = {np.dot(a, b):.4f}')
assert np.isclose(my_dot(a, b), np.dot(a, b))
print('✓ my_dot == np.dot')

# Norm 다리: u·u = ||u||_2^2
for x in [a, b]:
    print(f'  x·x = {my_dot(x, x):.4f},  ||x||_2^2 = {my_norm(x, 2) ** 2:.4f}')
print('✓ x·x = ||x||_2^2 (정의 2.2의 즉시 따름)')

# 각도 (정의 2.3)
cos_ab = my_dot(a, b) / (my_norm(a, 2) * my_norm(b, 2))
theta_ab = np.degrees(np.arccos(cos_ab))
print(f'\ncos(a, b) = {cos_ab:.6f}')
print(f'theta     = {theta_ab:.2f}도 → 둔각 (Dot product가 음수)')

## 5. Theorem: Cauchy-Schwarz 부등식 (정리 2.1)

### 정리 2.1 (Cauchy-Schwarz, $\mathbb{R}^n$)
모든 $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$에 대해
$$|\mathbf{u} \cdot \mathbf{v}| \;\le\; \|\mathbf{u}\|_2\,\|\mathbf{v}\|_2$$
등호 $\iff$ $\mathbf{u}, \mathbf{v}$가 **평행** (한쪽이 다른 쪽의 Scalar배).

강의교안 C-4의 증명 골자: $\|\mathbf{u} - t\mathbf{v}\|^2 \ge 0$이 모든 $t \in \mathbb{R}$에서 성립한다는 사실에서, $t$에 대한 이차식의 **판별식 $\le 0$**으로부터 따라옵니다.

여기서는 임의 Vector에서 부등식을 수치 검증하고, **평행일 때 등호**를 확인합니다.

In [ ]:
rng = np.random.default_rng(7)
n_trials = 1000
n = 5

viol = 0
max_gap = 0.0
for _ in range(n_trials):
    u = rng.normal(size=n)
    v = rng.normal(size=n)
    lhs = abs(my_dot(u, v))
    rhs = my_norm(u, 2) * my_norm(v, 2)
    if lhs > rhs + 1e-10:
        viol += 1
    max_gap = max(max_gap, rhs - lhs)

print(f'시행 {n_trials}회, R^{n} 임의 Vector')
print(f'  부등식 위반: {viol}회')
print(f'  최대 여유 (rhs - lhs): {max_gap:.4f}')
print('✓ |u·v| <= ||u|| ||v|| 모든 시행에서 성립')

# 등호 조건: v = c·u (평행)
u = np.array([1.0, 2.0, -1.0, 3.0])
v = 2.5 * u  # 평행 (c=2.5)
lhs = abs(my_dot(u, v))
rhs = my_norm(u, 2) * my_norm(v, 2)
print(f'\nv = 2.5·u (평행)일 때:')
print(f'  |u·v|       = {lhs:.6f}')
print(f'  ||u|| ||v|| = {rhs:.6f}')
print(f'  차이        = {abs(lhs - rhs):.2e}')
print('✓ 평행일 때 등호 성립')

## 6. Definition: Cosine similarity (정의 2.5)

### 정의 2.5 (Cosine similarity)
$\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$ ($\ne \mathbf{0}$)에 대해
$$\mathrm{cos\_sim}(\mathbf{u}, \mathbf{v}) := \cos\theta = \frac{\mathbf{u}\cdot\mathbf{v}}{\|\mathbf{u}\|_2\,\|\mathbf{v}\|_2} \;\in\; [-1, 1]$$

범위 $[-1, 1]$이 보장되는 이유 = **정리 2.1 Cauchy-Schwarz**.

### 핵심 성질: 스케일 불변성
모든 $\alpha, \beta > 0$에 대해
$$\mathrm{cos\_sim}(\alpha\mathbf{u}, \beta\mathbf{v}) = \mathrm{cos\_sim}(\mathbf{u}, \mathbf{v})$$
→ **방향만 보고 크기는 무시**. BERT·CLIP 같은 임베딩 비교의 표준 도구.

### 정의 2.4 (Orthogonal·직교)
$\mathbf{u}\cdot\mathbf{v} = 0$일 때 두 Vector를 **Orthogonal**이라 부릅니다 ($\theta = \pi/2$, $\cos\theta = 0$).

In [ ]:
def my_cosine_similarity(u, v):
    """Cosine similarity 직접 구현"""
    return my_dot(u, v) / (my_norm(u, 2) * my_norm(v, 2))


# 강의교안 D-2의 예시 재현: 같은 두 단어, 등장 횟수 10배
d1 = np.array([1.0, 1.0, 0.0, 0.0])
d2 = np.array([10.0, 10.0, 0.0, 0.0])

euclid = my_norm(d1 - d2, 2)
cos = my_cosine_similarity(d1, d2)
print(f'd1 = (1, 1, 0, 0),  d2 = (10, 10, 0, 0)')
print(f'  Euclidean 거리:  ||d1 - d2||_2 = {euclid:.4f}  → "멀다"')
print(f'  Cosine similarity:                {cos:.4f}  → "같다"')
print('\n→ 같은 두 단어만 들어 있는 두 문서, 길이 차이는 cosine으로 무시됩니다.')

# 스케일 불변성 수치 검증
u = np.array([1.0, 2.0, -1.0])
v = np.array([3.0, -2.0, 0.5])
base = my_cosine_similarity(u, v)
for alpha, beta in [(2, 5), (0.1, 100), (50, 0.01)]:
    scaled = my_cosine_similarity(alpha * u, beta * v)
    print(f'cos({alpha}u, {beta}v) = {scaled:.6f}  (base={base:.6f}, match={np.isclose(scaled, base)})')
print('✓ Cosine similarity는 양의 Scalar 스케일에 불변')

## 7. Application: MNIST 이미지 = $\mathbb{R}^{64}$ Vector

강의교안 D-1의 사실: **이미지 한 장 = $\mathbb{R}^d$ Vector**.

원본 MNIST는 28×28 → $\mathbb{R}^{784}$ Vector지만, 여기서는 다운로드가 필요 없는 `sklearn.datasets.load_digits()`의 **8×8 mini-MNIST → $\mathbb{R}^{64}$**를 사용합니다.

> 원본 28×28을 쓰려면 `tensorflow.keras.datasets.mnist.load_data()` 또는 `torchvision.datasets.MNIST`를 사용합니다.

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
X_img = digits.images   # (1797, 8, 8)
X = digits.data         # (1797, 64): 이미 flatten된 형태
y = digits.target       # (1797,)

print(f'이미지 개수      : {X_img.shape[0]}')
print(f'이미지 크기      : {X_img.shape[1]} x {X_img.shape[2]}')
print(f'Flatten Vector 차원: R^{X.shape[1]}')
print(f'레이블 종류      : {np.unique(y)}')

In [ ]:
# 두 이미지 시각화 + Cosine similarity
idx1, idx2 = 0, 100  # 임의의 두 이미지

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(X_img[idx1], cmap='gray_r')
axes[0].set_title(f'index={idx1}, label={y[idx1]}')
axes[0].axis('off')
axes[1].imshow(X_img[idx2], cmap='gray_r')
axes[1].set_title(f'index={idx2}, label={y[idx2]}')
axes[1].axis('off')
plt.show()

sim = my_cosine_similarity(X[idx1], X[idx2])
print(f'\ncos similarity(idx={idx1}, idx={idx2}) = {sim:.4f}')
print(f'두 라벨 같음? {y[idx1] == y[idx2]}')

## 8. Application: 같은 숫자 vs 다른 숫자 평균 Cosine (n=200)

한 쌍이 아니라 **모든 쌍의 평균**을 비교: 같은 라벨끼리가 다른 라벨끼리보다 cosine similarity가 평균적으로 더 높은가?

broadcasting + 행렬곱 한 줄(`X_norm @ X_norm.T`)로 모든 쌍의 cosine similarity 행렬을 얻습니다: 이 패턴이 **Part 3 7회차 Attention의 $QK^\top$**과 같습니다.

In [ ]:
# 작은 표본으로 평균 비교 (n=200으로 제한 → 200×200 행렬 안정)
n_sample = 200
X_s = X[:n_sample]
y_s = y[:n_sample]

# Cosine similarity 행렬 (broadcasting)
norms = np.sqrt((X_s ** 2).sum(axis=1, keepdims=True))  # (n, 1)
X_norm = X_s / (norms + 1e-12)                          # 행별 정규화
cos_mat = X_norm @ X_norm.T                             # (n, n)

# 같은 라벨 쌍 vs 다른 라벨 쌍
same_label = y_s[:, None] == y_s[None, :]
off_diag = ~np.eye(n_sample, dtype=bool)

same_mean = cos_mat[same_label & off_diag].mean()
diff_mean = cos_mat[~same_label].mean()

print(f'같은 라벨 쌍 평균 cos: {same_mean:.4f}')
print(f'다른 라벨 쌍 평균 cos: {diff_mean:.4f}')
print(f'차이                : {same_mean - diff_mean:.4f}')
print('→ 같은 숫자일수록 더 가깝다 (cosine으로 본 단순 분류기의 토대)')

## 9. Application: Top-5 Nearest Neighbor 검색

한 query 이미지가 주어졌을 때 **Cosine similarity가 가장 높은 상위 5개** 이미지를 찾고 시각화합니다.

강의교안 D-3에서 본 사실: 이것이 **RAG 문서 검색**·Word2Vec·CLIP의 핵심 패턴입니다.

In [ ]:
# query 선정
query_idx = 7
query_vec = X[query_idx]
query_label = y[query_idx]

# 모든 이미지와 query의 Cosine similarity 계산 (broadcasting)
norms_all = np.sqrt((X ** 2).sum(axis=1))
query_norm = np.sqrt((query_vec ** 2).sum())
sims = (X @ query_vec) / (norms_all * query_norm + 1e-12)

# 자기 자신 제외하고 top-5
sims_copy = sims.copy()
sims_copy[query_idx] = -np.inf
top5_idx = np.argsort(-sims_copy)[:5]

# 시각화
fig, axes = plt.subplots(1, 6, figsize=(12, 2.5))
axes[0].imshow(X_img[query_idx], cmap='gray_r')
axes[0].set_title(f'Query (label={query_label})')
axes[0].axis('off')
for ax, idx in zip(axes[1:], top5_idx):
    ax.imshow(X_img[idx], cmap='gray_r')
    ax.set_title(f'label={y[idx]}\ncos={sims[idx]:.3f}')
    ax.axis('off')
plt.tight_layout()
plt.show()

hit = (y[top5_idx] == query_label).sum()
print(f'\nQuery label {query_label} 기준 top-5 중 같은 라벨 {hit}/5')

## 10. 연습 (자가 점검)

다음을 코드 셀에 직접 시도해 보고 결과를 본인 노트에 기록합니다.

### 연습 1 (Top-5의 정확도)
`query_idx`를 0, 1, ..., 9로 바꿔가며 top-5 중 같은 라벨의 비율을 측정. 표로 정리합니다.

### 연습 2 (정규화 효과)
Cosine similarity 대신 **유클리드 거리** 기준으로 top-5를 찾아보고 결과 차이를 비교합니다.

### 연습 3 (Cauchy-Schwarz 등호 조건)
$\mathbf{u} \in \mathbb{R}^5$를 임의로 잡고 $\mathbf{v} = c \cdot \mathbf{u}$ ($c \in \mathbb{R}$)로 두면 Cauchy-Schwarz가 등호가 됨을 수치 확인하세요. $c$의 부호에 따라 cosine similarity가 어떻게 달라지는지 봅니다.

### 연습 4 (broadcasting vs for문 시간 비교)
$n = 10, 100, 1000$인 무작위 행렬 $X \in \mathbb{R}^{n \times 30}$에 대해
1. 이중 for문으로 $n \times n$ Cosine similarity 행렬 계산
2. broadcasting으로 같은 결과 계산 (위 §8의 `X_norm @ X_norm.T` 방식)

두 방식의 시간을 측정해 log-log 그래프로 그리고 broadcasting의 speedup을 확인합니다.

### 연습 5 (평행사변형 법칙 증명·검증)
임의의 $\mathbf{u}, \mathbf{v} \in \mathbb{R}^n$에 대해
$$\|\mathbf{u}+\mathbf{v}\|^2 + \|\mathbf{u}-\mathbf{v}\|^2 = 2(\|\mathbf{u}\|^2 + \|\mathbf{v}\|^2)$$
를 종이로 증명한 뒤 NumPy로 수치 확인하세요. (힌트: $\|\mathbf{w}\|^2 = \mathbf{w}\cdot\mathbf{w}$ + Dot product 분배.)

> 정답·풀이는 과제 2회차 제출본에 포함합니다.

## 11. 정리

오늘 도입한 정의·정리 목록:

| 번호 | 내용 |
|---|---|
| 정의 2.1 | Euclidean Norm $\|\mathbf{x}\|_2 = \sqrt{\sum x_i^2}$ + 일반화 $\ell_p$ |
| 정의 2.2 | Dot product $\mathbf{u}\cdot\mathbf{v} = \sum u_i v_i$ |
| 정리 2.1 | Cauchy-Schwarz: $\lvert \mathbf{u}\cdot\mathbf{v}\rvert \le \|\mathbf{u}\|\,\|\mathbf{v}\|$ |
| 정의 2.3 | 각도 $\cos\theta = \mathbf{u}\cdot\mathbf{v}/(\|\mathbf{u}\|\|\mathbf{v}\|)$ |
| 정의 2.4 | Orthogonal: $\mathbf{u}\cdot\mathbf{v} = 0$ |
| 정의 2.5 | Cosine similarity = $\cos\theta \in [-1, 1]$ |

오늘 검증한 사실:

- Norm·Dot product·Cosine similarity는 **NumPy 기본 연산만으로 한 줄씩 구현 가능**.
- Norm 세 성질 (N1·N2·N3)은 임의 Vector에서 모두 성립.
- 단위구 모양은 $p$에 따라 마름모·원·정사각형으로 달라짐.
- **Cauchy-Schwarz**가 cosine similarity의 범위 $[-1, 1]$과 $\ell_2$ 삼각부등식(triangle inequality)을 동시에 보장.
- Cosine similarity는 **스케일 불변**: 길이 다른 문서·이미지 비교에 자연스러움.
- MNIST에서 **같은 라벨끼리 cosine similarity가 평균적으로 더 높다**.
- broadcasting + 행렬곱(`X_norm @ X_norm.T`) 한 줄로 모든 쌍의 cosine 행렬을 얻을 수 있다: **Part 3 7회차 Attention의 $QK^\top$**과 같은 패턴.

### 다음 회차(3회차)로 가는 다리

오늘 본 Dot product가 그대로 **Matrix·Vector 곱의 Row picture 해석**이 됩니다. $A\mathbf{x}$의 $i$번째 성분 = $A$의 $i$행과 $\mathbf{x}$의 **Dot product**. 즉 오늘의 한 줄 정의 $\mathbf{u}\cdot\mathbf{v}$가 **모든 행에 일괄 적용된 것이 곧 행렬 곱**입니다. 1회차의 **Linear combination**은 같은 식의 **Column picture 해석**이 됩니다.